# GRAYMATTER : Full CISA (Production)

Trains the **Full CISA** variant (production model) across all 5 folds.

## Variant Details

| Parameter | Value |
|-----------|-------|
| `skip_mode` | `full` |
| Description | Full CISA: coordinate gating + axial inter-slice context |
| Estimated Time | ~7-8 hours |

## Setup

1. Run all cells

Dataset location is resolved automatically.

In [ ]:
# Install dependencies
import importlib.util, subprocess, sys
for pkg in ["monai", "nibabel", "tqdm"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Dependencies ready!")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
import json, os, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# --- Paths (auto-resolved; not printed) ---
def _resolve_dataset_dir() -> Path:
    kaggle_input = Path("/kaggle/input")
    candidates = []
    if kaggle_input.exists():
        candidates.extend(sorted(kaggle_input.glob("graymatter-dataset*")))
        candidates.extend(sorted(p for p in kaggle_input.iterdir() if p.is_dir()))
    candidates.extend([Path("dataset"), Path("../dataset"), Path("/kaggle/working/dataset")])
    for cand in candidates:
        if cand.is_dir() and (cand / "manifests").is_dir():
            return cand
    raise FileNotFoundError(
        "Dataset with manifests/ not found. Attach the MONAI hippocampus dataset "
        "or place dataset/ next to the notebook."
    )

DATASET_DIR = _resolve_dataset_dir()
OUTPUT_DIR = Path("ai/results/ablations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- This notebook runs ONLY full_cisa ---
VARIANT = "full_cisa"
SKIP_MODE = "full"
ALL_FOLDS = [1, 2, 3, 4, 5]

# --- Training config ---
MAX_EPOCHS = 300
EARLY_STOP = 30
BATCH_SIZE = 2
LR = 5e-4
ROI = (48, 64, 48)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Variant: {VARIANT} (skip_mode={SKIP_MODE})")
print(f"Folds: {ALL_FOLDS}")
print(f"Device: {device}")
if device.type == "cuda":

In [ ]:
# =============================================================================
# MODEL ARCHITECTURE
# =============================================================================
from typing import Literal, Sequence
import torch.nn as nn

def _gn(ch):
    g = min(8, ch)
    while g > 1 and ch % g != 0: g -= 1
    return nn.GroupNorm(g, ch)

class DoubleConv3D(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(nn.Conv3d(i,o,3,padding=1,bias=False),_gn(o),nn.ReLU(True),nn.Conv3d(o,o,3,padding=1,bias=False),_gn(o),nn.ReLU(True))
    def forward(self, x): return self.b(x)

class CISA(nn.Module):
    def __init__(self, ch, mode="full"):
        super().__init__()
        self.mode = mode
        if mode == "identity": return
        mid = max(ch // 8, 8)
        def br(): return nn.Sequential(nn.Conv3d(ch,mid,1,bias=False),nn.ReLU(True),nn.Conv3d(mid,ch,1,bias=False))
        self.bd,self.bh,self.bw = br(),br(),br()
        self.sig = nn.Sigmoid()
        if mode == "full":
            self.inter = nn.Sequential(nn.Conv3d(ch,ch,(3,1,1),padding=(1,0,0),groups=ch,bias=False),_gn(ch),nn.ReLU(True))
        else: self.inter = None
    def forward(self, x):
        if self.mode == "identity": return x
        g = x * self.sig(self.bd(x.mean(2,keepdim=True)))
        g = g * self.sig(self.bh(g.mean(3,keepdim=True)))
        g = g * self.sig(self.bw(g.mean(4,keepdim=True)))
        if self.mode == "coord_only": return g
        return g + self.inter(g)

class UNet3D(nn.Module):
    def __init__(self, skip_mode="full"):
        super().__init__()
        c = [32,64,128,256]
        self.e1,self.e2,self.e3,self.e4 = [DoubleConv3D(c[i-1] if i else 1, c[i]) for i in range(4)]
        self.pool = nn.MaxPool3d(2)
        self.bn = nn.Sequential(DoubleConv3D(256,256), nn.Dropout3d(0.1))
        self.s4,self.s3,self.s2,self.s1 = [CISA(c[i], mode=skip_mode) for i in range(3,-1,-1)]
        self.u4,self.u3,self.u2,self.u1 = nn.ConvTranspose3d(c[3],c[3],2,2),nn.ConvTranspose3d(c[3],c[2],2,2),nn.ConvTranspose3d(c[2],c[1],2,2),nn.ConvTranspose3d(c[1],c[0],2,2)
        self.d4 = DoubleConv3D(512,256)
        self.d3 = DoubleConv3D(256,128)
        self.d2 = DoubleConv3D(128,64)
        self.d1 = DoubleConv3D(64,32)
        self.out = nn.Conv3d(32,3,1)
    def forward(self, x):
        e1=self.e1(x); e2=self.e2(self.pool(e1)); e3=self.e3(self.pool(e2)); e4=self.e4(self.pool(e3))
        b=self.bn(self.pool(e4))
        d4=self.d4(torch.cat([self.u4(b),self.s4(e4)],1))
        d3=self.d3(torch.cat([self.u3(d4),self.s3(e3)],1))
        d2=self.d2(torch.cat([self.u2(d3),self.s2(e2)],1))
        d1=self.d1(torch.cat([self.u1(d2),self.s1(e1)],1))
        return self.out(d1)

print(f"Model loaded: {VARIANT}")

In [ ]:
# =============================================================================
# TRANSFORMS & DATALOADERS
# =============================================================================
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, ScaleIntensityRanged,
    SpatialPadd, ResizeWithPadOrCropd, EnsureTyped, RandFlipd, AsDiscreted,
)
from monai.data import CacheDataset, DataLoader, list_data_collate

def train_tf():
    return Compose([
        LoadImaged(keys=["image","label"], image_only=False),
        EnsureChannelFirstd(keys=["image","label"]),
        Orientationd(keys=["image","label"], axcodes="RAS"),
        ScaleIntensityRanged(keys=["image"], a_min=0, a_max=2500, b_min=0, b_max=1, clip=True),
        AsDiscreted(keys=["label"]),
        SpatialPadd(keys=["image","label"], spatial_size=ROI),
        ResizeWithPadOrCropd(keys=["image","label"], spatial_size=ROI),
        EnsureTyped(keys=["image","label"], track_meta=False),
        RandFlipd(keys=["image","label"], spatial_axis=0, prob=0.5),
        RandFlipd(keys=["image","label"], spatial_axis=1, prob=0.5),
        RandFlipd(keys=["image","label"], spatial_axis=2, prob=0.5),
    ])

def val_tf():
    return Compose([
        LoadImaged(keys=["image","label"], image_only=False),
        EnsureChannelFirstd(keys=["image","label"]),
        Orientationd(keys=["image","label"], axcodes="RAS"),
        ScaleIntensityRanged(keys=["image"], a_min=0, a_max=2500, b_min=0, b_max=1, clip=True),
        AsDiscreted(keys=["label"]),
        SpatialPadd(keys=["image","label"], spatial_size=ROI),
        ResizeWithPadOrCropd(keys=["image","label"], spatial_size=ROI),
        EnsureTyped(keys=["image","label"]),
    ])

def find_file(base_dir, filename):
    p = base_dir / filename
    if p.exists(): return p
    if filename.endswith('.nii.gz'):
        p2 = base_dir / filename.replace('.nii.gz', '.nii')
        if p2.exists(): return p2
    raw_dir = DATASET_DIR / 'raw' / base_dir.relative_to(DATASET_DIR)
    p3 = raw_dir / filename
    if p3.exists(): return p3
    if filename.endswith('.nii.gz'):
        p4 = raw_dir / filename.replace('.nii.gz', '.nii')
        if p4.exists(): return p4
    return p

def load_fold(n):
    with open(DATASET_DIR / "manifests" / f"fold{n}.json") as f:
        manifest = json.load(f)
    tr = []
    for c in manifest["training"]["cases"]:
        rel = Path(c["image"]).relative_to("dataset")
        img = find_file(DATASET_DIR / rel.parent, rel.name)
        rel = Path(c["label"]).relative_to("dataset")
        lbl = find_file(DATASET_DIR / rel.parent, rel.name)
        tr.append({"image": str(img), "label": str(lbl)})
    va = []
    for c in manifest["validation"]["cases"]:
        rel = Path(c["image"]).relative_to("dataset")
        img = find_file(DATASET_DIR / rel.parent, rel.name)
        rel = Path(c["label"]).relative_to("dataset")
        lbl = find_file(DATASET_DIR / rel.parent, rel.name)
        va.append({"image": str(img), "label": str(lbl)})
    return tr, va

print("Transforms ready!")

In [ ]:
# =============================================================================
# TRAINING FUNCTIONS
# =============================================================================
from monai.losses import DiceCELoss
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm
import random

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def get_dice(pred, lbl, num_classes=3):
    dices = []
    for c in range(1, num_classes):
        p = (pred == c).float().flatten()
        t = (lbl == c).float().flatten()
        inter = (p * t).sum()
        dices.append((2*inter / (p.sum()+t.sum()+1e-8)).item())
    return np.mean(dices)

def train_fold(fold_num):
    fold_dir = OUTPUT_DIR / VARIANT / f"fold{fold_num}"
    ckpt_path = fold_dir / "best_model.pth"
    
    # Skip if already done
    if ckpt_path.exists():
        with open(fold_dir / "eval_metrics.json") as f:
            prev = json.load(f)
        print(f"  ✓ Fold {fold_num} already done (DSC: {prev['best_dice']:.4f}) - SKIPPING")
        return prev
    
    fold_dir.mkdir(parents=True, exist_ok=True)
    set_seed(42 + fold_num)
    
    # Load data
    train_cases, val_cases = load_fold(fold_num)
    train_ds = CacheDataset(train_cases, train_tf(), cache_rate=1.0, num_workers=0)
    val_ds = CacheDataset(val_cases, val_tf(), cache_rate=1.0, num_workers=0)
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=list_data_collate)
    val_loader = DataLoader(val_ds, 1)
    
    # Model
    model = UNet3D(skip_mode=SKIP_MODE).to(device)
    weights = torch.tensor([0.167, 2.83, 3.07], device=device)
    loss_fn = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True, weight=weights, lambda_dice=1.5, lambda_ce=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=10)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=290)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], [10])
    scaler = GradScaler()
    
    best_dice = 0.0
    patience = 0
    history = []
    start = time.time()
    
    for epoch in range(MAX_EPOCHS):
        # Train
        model.train()
        tloss = 0
        for batch in train_loader:
            imgs, lbls = batch["image"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            with autocast("cuda"):
                loss = loss_fn(model(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            tloss += loss.item()
        scheduler.step()
        tloss /= len(train_loader)
        
        # Validate
        model.eval()
        vdices = []
        with torch.no_grad():
            for batch in val_loader:
                imgs, lbls = batch["image"].to(device), batch["label"].to(device)
                out = model(imgs).argmax(dim=1, keepdim=True)
                vdices.append(get_dice(out, lbls))
        vdice = np.mean(vdices)
        lr = optimizer.param_groups[0]["lr"]
        
        history.append({"epoch": epoch+1, "train_loss": tloss, "val_dice": vdice, "lr": lr})
        
        if vdice > best_dice:
            best_dice = vdice
            patience = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "dice": best_dice}, ckpt_path)
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f"    Epoch {epoch+1:3d} | Loss: {tloss:.4f} | DSC: {vdice:.4f} | LR: {lr:.6f} [BEST]")
        else:
            patience += 1
            if (epoch+1) % 25 == 0:
                print(f"    Epoch {epoch+1:3d} | Loss: {tloss:.4f} | DSC: {vdice:.4f} | Pat: {patience}/{EARLY_STOP}")
        
        if patience >= EARLY_STOP:
            print(f"    Early stopping at epoch {epoch+1}")
            break
    
    elapsed = time.time() - start
    
    # Save results
    pd.DataFrame(history).to_csv(fold_dir / "training_history.csv", index=False)
    result = {
        "fold": fold_num, "variant": VARIANT, "skip_mode": SKIP_MODE,
        "best_dice": best_dice, "epochs": len(history), "time_min": elapsed/60,
    }
    with open(fold_dir / "eval_metrics.json", "w") as f:
        json.dump(result, f, indent=2)
    
    print(f"    ✓ Done: Best DSC = {best_dice:.4f}, Time = {elapsed/60:.1f} min")
    return result

print("Training functions ready!")

In [ ]:
# =============================================================================
# RUN TRAINING
# =============================================================================
print("="*70)
print(f"GRAYMATTER ABLATION: {VARIANT.upper()}")
print(f"Skip mode: {SKIP_MODE}")
print(f"Folds: {ALL_FOLDS}")
print("="*70)

# Check what's already done
completed = sum(1 for f in ALL_FOLDS if (OUTPUT_DIR / VARIANT / f"fold{f}" / "best_model.pth").exists())
print(f"\nCompleted: {completed}/{len(ALL_FOLDS)} folds")

all_results = []
session_start = time.time()
MAX_SESSION_TIME = 11 * 3600  # 11 hours (leave buffer)

for fold_num in ALL_FOLDS:
    # Check time limit
    elapsed = time.time() - session_start
    if elapsed > MAX_SESSION_TIME:
        print(f"\n⚠ Session time limit reached ({elapsed/3600:.1f} hrs). Run again to continue.")
        break
    
    remaining = (MAX_SESSION_TIME - elapsed) / 60
    print(f"\nFold {fold_num}/5 ({remaining:.0f} min remaining)")
    
    try:
        result = train_fold(fold_num)
        all_results.append(result)
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        continue

# Save variant summary
if all_results:
    dices = [r["best_dice"] for r in all_results]
    summary = {
        "variant": VARIANT, "skip_mode": SKIP_MODE,
        "folds_done": len(all_results), "mean_dice": float(np.mean(dices)),
        "std_dice": float(np.std(dices)), "fold_results": all_results,
    }
    variant_dir = OUTPUT_DIR / VARIANT
    variant_dir.mkdir(parents=True, exist_ok=True)
    with open(variant_dir / "cv_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n{'='*70}")
    print(f"{VARIANT} SUMMARY")
    print(f"Mean DSC: {summary['mean_dice']:.4f} ± {summary['std_dice']:.4f}")
    print(f"Folds completed: {summary['folds_done']}/{len(ALL_FOLDS)}")
    print(f"{'='*70}")

In [ ]:
# =============================================================================
# RESULTS
# =============================================================================
if all_results:
    print("\n" + "="*70)
    print("FOLD RESULTS")
    print("="*70)
    print(f"{'Fold':<8} {'Best DSC':<12} {'Epochs':<10} {'Time (min)':<12}")
    print("-"*70)
    for r in all_results:
        print(f"{r['fold']:<8} {r['best_dice']:<12.4f} {r['epochs']:<10} {r['time_min']:<12.1f}")
    print("-"*70)
    
    dices = [r["best_dice"] for r in all_results]
    print(f"\nMean: {np.mean(dices):.4f} ± {np.std(dices):.4f}")
    print(f"Range: [{np.min(dices):.4f}, {np.max(dices):.4f}]")

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("1. Run the other 2 variant notebooks:")
print("   - graymatter-ablation-plain-unet.ipynb")
print("   - graymatter-ablation-coord-attention.ipynb")
print("2. Download all results from Output tab")
print("3. Copy to ai/results/ablations/")
print("4. Run: python paper/scripts/plot_ablation_comparison.py")